# Stability Analysis

Three questions:
1. **Neuron flip rate** — within a single free-phase run, how many neurons keep changing sign? Goes to 0 at a fixed point.
2. **State drift** — does the free-phase endpoint C change between consecutive weight updates? Tracks whether the weight landscape has converged.
3. **Weight update norm** — does |ΔW| decay over time? Detects whether learning is slowing down.

All metrics shown per rule (current, approximate, long_warmup, EP α=0.3), mean ± std over 3 seeds.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

RESULTS_FILE = Path('../results/stability/stability_results.json')
FIGURES_DIR  = Path('../figures/stability')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_FILE) as f:
    data = json.load(f)

results = data['results']
available_rules = sorted(set(r['rule'] for r in results))
print('Rules in file:', available_rules)
print('Seeds per rule:', {rule: sum(1 for r in results if r['rule'] == rule) for rule in available_rules})
print('Steps per run:', len(results[0]['trajectory']))

In [ ]:
RULE_COLORS = {
    'current':      '#1f77b4',
    'approximate':  '#ff7f0e',
    'long_warmup':  '#2ca02c',
    'ep':           '#d62728',
}
RULE_LABELS = {
    'current':     'Current (Rule 1)',
    'approximate': 'Approximate (Rule 2)',
    'long_warmup': 'Long Warmup (Rule 4)',
    'ep':          'EP α=0.3 (Rule 3)',
}

def get_metric(results, rule, key, subkey=None):
    """Extract trajectory of a scalar metric across weight-update steps, grouped by seed."""
    seeds = [r for r in results if r['rule'] == rule]
    all_vals = []
    for r in seeds:
        vals = []
        for step in r['trajectory']:
            v = step[key] if subkey is None else step[key][subkey]
            vals.append(v if v is not None else np.nan)
        all_vals.append(vals)
    return np.array(all_vals)  # shape (n_seeds, n_steps)

def plot_mean_std(ax, xs, arr, color, label):
    """Plot mean ± std shading; arr shape (n_seeds, n_steps)."""
    mean = np.nanmean(arr, axis=0)
    std  = np.nanstd(arr, axis=0)
    ax.plot(xs, mean, color=color, label=label, linewidth=2)
    ax.fill_between(xs, mean - std, mean + std, color=color, alpha=0.15)

## Figure 1 — Neuron Flip Rate (at last free-phase step)

Fraction of neurons that still flip their sign at the final dynamic step of the free phase.
**If this is 0, the network has reached a fixed point within the free phase.**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for rule in available_rules:
    arr = get_metric(results, rule, 'flip_rate_final')
    xs  = np.arange(arr.shape[1])
    plot_mean_std(ax, xs, arr, RULE_COLORS[rule], RULE_LABELS.get(rule, rule))

ax.set_xlabel('Weight update step', fontsize=12)
ax.set_ylabel('Neuron flip rate (at final free step)', fontsize=12)
ax.set_title('Fig 1 — Does the network reach a fixed point within the free phase?', fontsize=13)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.legend()
ax.set_ylim(bottom=-0.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'stability_fig1_flip_rate.png', dpi=150)
plt.show()
print('Saved fig1')

## Figure 2 — Convergence Curve (flip rate across dynamic steps, sampled at K=0, 30, 59)

Within a single free-phase run of 20 dynamic steps, how quickly do neurons stop flipping?
We plot this curve at the beginning, middle, and end of training to see how it evolves.

In [ ]:
n_steps = len(results[0]['trajectory'])
checkpoints = [0, n_steps // 2, n_steps - 1]

fig, axes = plt.subplots(1, len(available_rules), figsize=(5 * len(available_rules), 4), sharey=True)
if len(available_rules) == 1:
    axes = [axes]

for ax, rule in zip(axes, available_rules):
    seeds = [r for r in results if r['rule'] == rule]
    for k_idx, k in enumerate(checkpoints):
        curves = np.array([r['trajectory'][k]['flip_curve'] for r in seeds])  # (n_seeds, n_dyn_steps)
        mean = np.mean(curves, axis=0)
        std  = np.std(curves, axis=0)
        xs = np.arange(len(mean))
        alpha = 0.4 + 0.3 * k_idx
        c = RULE_COLORS.get(rule, 'black')
        ax.plot(xs, mean, color=c, alpha=alpha, linewidth=2, label=f'after {k} updates')
        ax.fill_between(xs, mean - std, mean + std, color=c, alpha=0.08)
    ax.set_title(RULE_LABELS.get(rule, rule), fontsize=11)
    ax.set_xlabel('Dynamic step within free phase')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Neuron flip rate')
fig.suptitle('Fig 2 — Convergence curve within free phase at different training stages', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'stability_fig2_convergence_curve.png', dpi=150)
plt.show()
print('Saved fig2')

## Figure 3 — State Drift Between Consecutive Weight Updates

How much does state C (the training endpoint) change between update k and k+1?
If drift → 0, the weight updates are no longer moving the state — the weight landscape has converged.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for rule in available_rules:
    arr = get_metric(results, rule, 'state_drift_fc')
    # step 0 is NaN (no prev state); start from step 1
    arr = arr[:, 1:]
    xs  = np.arange(1, arr.shape[1] + 1)
    plot_mean_std(ax, xs, arr, RULE_COLORS[rule], RULE_LABELS.get(rule, rule))

ax.set_xlabel('Weight update step', fontsize=12)
ax.set_ylabel('State C drift (L2 / √N)', fontsize=12)
ax.set_title('Fig 3 — Does state C stabilise across weight updates?', fontsize=13)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.legend()
ax.set_ylim(bottom=-0.02)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'stability_fig3_state_drift.png', dpi=150)
plt.show()
print('Saved fig3')

## Figure 4 — Weight Update Norm |ΔW|

Does the learning slow down over time? If weights converge to a solution, |ΔW| should decrease.
Shown for total norm and per-weight-matrix (W_in, J_conv, J_fc, W_out).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: total norm per rule
ax = axes[0]
for rule in available_rules:
    arr = get_metric(results, rule, 'weight_norms', 'total')
    xs  = np.arange(arr.shape[1])
    plot_mean_std(ax, xs, arr, RULE_COLORS[rule], RULE_LABELS.get(rule, rule))
ax.set_xlabel('Weight update step', fontsize=12)
ax.set_ylabel('|ΔW| total (L2)', fontsize=12)
ax.set_title('Total weight update norm', fontsize=12)
ax.legend()

# Right: per-matrix breakdown for long_warmup (most interesting)
ax = axes[1]
rule_focus = 'long_warmup' if 'long_warmup' in available_rules else available_rules[0]
component_colors = {'w_in': '#1f77b4', 'j_conv': '#ff7f0e', 'j_fc': '#2ca02c', 'w_out': '#d62728'}
for comp, color in component_colors.items():
    arr = get_metric(results, rule_focus, 'weight_norms', comp)
    xs  = np.arange(arr.shape[1])
    plot_mean_std(ax, xs, arr, color, comp)
ax.set_xlabel('Weight update step', fontsize=12)
ax.set_ylabel('|ΔW| component (L2)', fontsize=12)
ax.set_title(f'Per-matrix breakdown — {RULE_LABELS.get(rule_focus, rule_focus)}', fontsize=12)
ax.legend()

fig.suptitle('Fig 4 — Weight update norms over training', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'stability_fig4_weight_norms.png', dpi=150)
plt.show()
print('Saved fig4')

## Figure 5 — Summary: All Three Stability Metrics Side by Side

One compact figure combining flip rate, state drift, and weight norm for a direct comparison.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

titles = [
    ('flip_rate_final', None, 'Neuron flip rate\n(at final free step)'),
    ('state_drift_fc',  None, 'State C drift (L2/√N)\nbetween updates'),
    ('weight_norms',  'total', 'Weight update norm |ΔW|'),
]

for ax, (key, subkey, ylabel) in zip(axes, titles):
    for rule in available_rules:
        arr = get_metric(results, rule, key, subkey)
        if key == 'state_drift_fc':
            arr = arr[:, 1:]
            xs = np.arange(1, arr.shape[1] + 1)
        else:
            xs = np.arange(arr.shape[1])
        plot_mean_std(ax, xs, arr, RULE_COLORS[rule], RULE_LABELS.get(rule, rule))
    ax.set_xlabel('Weight update step')
    ax.set_ylabel(ylabel)
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.7)
    ax.legend(fontsize=7)

fig.suptitle('Fig 5 — Stability analysis: three metrics, all rules', fontsize=13)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'stability_fig5_summary.png', dpi=150)
plt.show()
print('Saved fig5')

## Interpretation

**What to look for:**

- **Flip rate = 0** → the free phase has reached a true fixed point. Long warmup should show this from early on (that's why CD=1.0). Current rule may still have nonzero flip rate late in training, consistent with the residual gap.

- **State drift = 0** → the training endpoint C is not moving between updates. This is the weight-landscape convergence signal. If drift is still large at K=60, the network hasn't settled.

- **|ΔW| decreasing** → the local plasticity rule is effectively reducing the violation count. If |ΔW| stays constant or grows, it means the rule keeps finding neurons to update regardless of training progress — no self-limiting mechanism.

**Expected pattern:** long_warmup should show flip_rate=0 throughout (states are fixed points by construction), near-zero drift, and |ΔW| that slowly decays. Current rule likely shows persistent nonzero flip rate, larger drift, and |ΔW| that doesn't decay — consistent with the inference gap never closing to 1.0.